# P117 — AgentBench: evaluar modelos de lenguaje como agentes

## 1. Título y paper

**Paper:** *AgentBench: Evaluating LLMs as Agents*  
**Autoría:** Xiao Liu, Hao Yu, Hanchen Zhang, Yifan Xu, Xuanyu Lei, y otros  
**Año y venue:** 2023 · arXiv:2308.03688 · ICLR 2024  
**Nivel:** L3 · **Motor:** `agentops`  
**Ficha completa:** [`P117_agentops`](../../papers/foundational/P117_agentops/README.md)

**Hito:** Evalúa agentes en ocho entornos distintos y hace visible que la tasa agregada esconde dónde y cómo fallan.

- [arXiv:2308.03688](https://arxiv.org/abs/2308.03688)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Los agentes se anunciaban con una cifra global de éxito. Esa cifra no dice en qué entornos sirven, en qué paso se pierden ni por qué modo fallan, que es exactamente lo que hace falta para operarlos y para decidir qué arreglar.
2. Ejecutar una implementación mínima de la propuesta: Un banco de pruebas multi-entorno —sistema operativo, base de datos, grafo de conocimiento, juegos, compras, navegación web— con evaluación por trayectoria y análisis de los modos de fallo, no solo de la tasa final.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P104
- P106
- P16


## 4. Intuición

«El agente resuelve el 35 % de las tareas». Ese número no dice en qué entornos sirve, en qué paso se pierde ni por qué modo falla — que es justo lo que hace falta para operarlo y para saber qué arreglar.


## 5. Concepto mínimo

```text
De la tasa agregada a la trayectoria:
  · tasa POR ENTORNO         ¿dónde sirve y dónde no?
  · MODO de fallo            ¿qué hay que arreglar?
  · PASO del fallo           ¿casi lo consigue o se pierde al principio?
  · LONGITUD de la trayectoria  ¿se puede cortar antes de saber el resultado?
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('agentops', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuánto varía la tasa entre entornos?
2. ¿Cuál es el modo de fallo dominante?
3. ¿Sirve la longitud de la trayectoria como señal?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('agentops', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('agentops', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

La tasa global es **0,35** y por entorno va de **0,16** a **0,565**: hay entornos donde el agente sencillamente no sirve. El modo dominante es «formato de llamada inválido» con **19 de 78** fallos. Y sí sirve la longitud: los episodios con éxito duran **7,9 pasos** de media y los que entran en bucle repetitivo, **26,1**.


## 10. Comentario pedagógico

Esa última cifra es la más útil en operación: permite **cortar antes de saber el resultado**. Un agente cuya trayectoria se alarga por encima de la distribución de los éxitos casi seguro está en un bucle, y cada paso extra cuesta dinero. Un límite de pasos no es una restricción arbitraria: es el mecanismo que convierte un fallo en un fallo barato.


## 11. Error o anti-patrón deliberado

Anti-patrón: operar un agente vigilando solo su tasa de éxito.


In [ ]:
print('La tasa llega tarde: se sabe cuando el episodio termino y ya se gasto todo.')
print('En operacion hay que vigilar la trayectoria: longitud, repeticiones, herramientas.')
print('Son las senales que permiten cortar a tiempo.')

## 12. Corrección

Lo que hay que instrumentar:


In [ ]:
r = run_paper_lab('agentops', seed=7)['result']
print('tasa global:', r['tasa_de_exito_global'])
print('por entorno:', r['por_entorno'])
print('modos de fallo:', r['modos_de_fallo'])
print('pasos medios exito/fallo:', r['pasos_medios_exito'], '/', r['pasos_medios_fallo'])

## 13. Desafío guiado

Explica por qué saber el modo de fallo dominante cambia qué se arregla primero, y qué se arreglaría mirando solo la tasa.


In [ ]:
r = run_paper_lab('agentops', seed=3)['result']
show(r)

## 14. Desafío autónomo

Instrumenta un agente tuyo para registrar por trayectoria: entorno, número de pasos, herramientas invocadas y motivo de terminación. Analiza veinte trayectorias fallidas y clasifica sus modos.


## 15. Evidencia de aprendizaje

Guarda el desglose por entorno y por modo de fallo, con tu criterio de corte por longitud de trayectoria.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P117_agentops/README.md) · evaluación formal: [`assessments/papers/P117_agentops.md`](../../assessments/papers/P117_agentops.md)


## 16. Cierre

Aquí se cierra la ruta de operación y con ella la segunda tanda: fundamentos probabilísticos, sistemas encarnados y la ingeniería que los sostiene en producción.


## 17. Conexión con el siguiente hito



Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
